In [ ]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [ ]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("threat_analysis")
    .getOrCreate()
    )

In [ ]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

## Validação da Ameaça

In [ ]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
df_threat = spark.read.parquet(threat_dataset_path)

In [ ]:
def plot_threat_event(df_threat, show_player_names=True):
    """
    Plota um evento para validar:
    - attackers_between_ball_goal
    - defenders_between_ball_goal
    - total_players_between_ball_goal
    - atk_def_advantage_between_ball_goal

    Espera um DataFrame Spark contendo exatamente um evento.
    """

    pdf = df_threat.toPandas()

    if len(pdf) != 1:
        raise ValueError("O dataframe deve conter exatamente um evento.")

    row = pdf.iloc[0]

    attackers = row["attackingPlayersNorm"]
    defenders = row["defendingPlayersNorm"]
    ball = row["ballsNorm"][0]

    stadium_length = row["stadiumLength"]
    stadium_width = row["stadiumWidth"]

    left_x = -stadium_length / 2
    right_x = stadium_length / 2
    bottom_y = -stadium_width / 2
    top_y = stadium_width / 2

    ball_x = ball["x"]

    fig = go.Figure()

    # ============================
    # Atacantes
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in attackers],
        y=[p["y"] for p in attackers],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in attackers] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "red" if p["x"] >= ball_x else "lightcoral"
                for p in attackers
            ]
        ),
        name="Attackers"
    )
)

    # ============================
    # Defensores
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in defenders],
        y=[p["y"] for p in defenders],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in defenders] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "blue" if p["x"] >= ball_x else "lightblue"
                for p in defenders
            ]
        ),
        name="Defenders"
    )
)
    
    # ============================
    # Bola
    # ============================

    fig.add_trace(
        go.Scatter(
            x=[ball["x"]],
            y=[ball["y"]],
            mode="markers",
            marker=dict(
                color="black",
                size=10,
                symbol="circle"
            ),
            name="Ball"
        )
    )

    # ============================
    # Linha da bola
    # ============================

    fig.add_vline(
        x=ball_x,
        line_dash="dash",
        line_width=2,
        line_color="black"
    )

    # ============================
    # Limites do campo
    # ============================

    fig.update_xaxes(
        range=[left_x, right_x],
        title="X",
        zeroline=False
    )

    fig.update_yaxes(
        range=[bottom_y, top_y],
        title="Y",
        scaleanchor="x",
        scaleratio=1,
        zeroline=False
    )
    
    # ============================
    # Contorno do campo
    # ============================

    fig.add_shape(
        type="rect",
        x0=left_x,
        y0=bottom_y,
        x1=right_x,
        y1=top_y,
        line=dict(
            color="black",
            width=2
        ),
        fillcolor="rgba(0,0,0,0)"
    )

    # ============================
    # Layout
    # ============================
    height = 600
    width = int(height * stadium_length / stadium_width)
    
    fig.update_layout(
    template="simple_white",
    width=width,
    height=height,
    margin=dict(l=20, r=120, t=60, b=20),
    title=(
        f"Attacking: {row['eventTeamName']} | "
        #f"attackingDirection: {row['attackingDirection']} | "       
        #f"Attackers: {row['attackers_between_ball_goal']} | "
        #f"Defenders: {row['defenders_between_ball_goal']} | "
        f"Total: {row['total_players_between_ball_goal']} | "
        f"Advantage: {row['atk_def_advantage_between_ball_goal']} | "
        f"Progression: {row['progression_distance']} | "
        f"Threat: {row['threat_score']:.3f}"
    ),
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.8)"
    )
)

    fig.show()

In [ ]:
df_threat.select('progression_distance').summary().show()

In [ ]:
df_threat_MC = df_threat.filter(F.col('homeTeamName') == 'Manchester City')

In [ ]:
df_threat_MC.select('progression_distance').summary().show()

In [ ]:
df_threat_MC.select('date', 
                    'gameId', 
                    'startFormattedGameClock',
                    'eventId', 
                    'eventTypeDescription',                    
                    'homeTeam', 
                    'HomeTeamName',
                    'opponentTeamName',
                    'attackers_between_ball_goal',
                    'defenders_between_ball_goal',
                    'progression_distance',
                    'total_players_between_ball_goal',
                    'atk_def_advantage_between_ball_goal',
                    'progression_distance_norm',
                    'total_players_between_ball_goal_norm',
                    'atk_def_advantage_between_ball_goal_norm',
                    'threat_score'
                    ).show(1000, truncate=False)

In [ ]:
# chute do time da casa
event_id = "3a725c404b084914d6f1fef150fb77f9"

df_threat.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat=df_threat.filter(F.col("eventId") == event_id), show_player_names=True)

# attacking players e defending correto mas está atacando pra esquerda

In [ ]:
# clearence do time adversário
event_id = 'c9cf2aa379ba29c9b804597e70e7a202'

df_threat.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat=df_threat.filter(F.col("eventId") == event_id), show_player_names=True)

# attacking e defending team estão corretos mas está atacando pra esquerda

In [ ]:
# cross na posse do time adversário
event_id = '05295389fbfeb8cb3226f4610b8ca26a'

df_threat.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat=df_threat.filter(F.col("eventId") == event_id), show_player_names=True)

#### 1.2. Preparação da base de odds

In [ ]:
# base de odds da Premier League 2022-2023
match_stats_22_23_path = str(data_folder_path / "match_stats" / "PL_22_23.csv")
df_pl_match_stats_22_23 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(match_stats_22_23_path , sep=',')

In [ ]:
df_pl_match_stats_22_23_filtrado = df_pl_match_stats_22_23.select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("date"),
    F.col('HomeTeam').alias('homeTeamName'),
    F.col('AwayTeam').alias('opponentTeamName'),
    'FTHG',
    'FTAG',
    'FTR',
    #'HTHG',
    #'HTAG',
    #'HTR',
    'HS',
    'AS',
    'HST',
    'AST',    
    'AvgH',
    'AvgA',
    'AvgD'
).sort('date')

In [ ]:
team_name_mapping = {
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton & Hove Albion",
    "Man City": "Manchester City",
    "Crystal Palace": "Crystal Palace",
    "Leicester": "Leicester City",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Fulham": "Fulham",
    "West Ham": "West Ham",
    "Man United": "Manchester United",
    "Wolves": "Wolverhampton Wanderers",
    "Southampton": "Southampton",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Everton": "Everton",
    "Leeds": "Leeds United",
    "Arsenal": "Arsenal",
    "Brentford": "Brentford"
}

df_pl_match_stats_22_23_filtrado_mapped = (
    df_pl_match_stats_22_23_filtrado
    .replace(team_name_mapping, subset=["homeTeamName", "opponentTeamName"])
)

df_pl_match_stats_22_23_filtrado_mapped.show()

### 2. Criação das bases agregadas e teste de correlação

In [ ]:
df_avg_threat = (
    df_threat
    .groupBy(
        "gameId",
        "competitionId",
        "season",
        "date",
        'homeTeamName', 
        'opponentTeamName',
        "homeTeam",
    )
    .agg(
        F.round(F.mean(F.col("threat_score")), 3).alias("avg_threat_score"),
        F.round(F.mean(F.col('attackers_between_ball_goal')), 2).alias('avg_attackers_between_ball_goal'),
        F.round(F.mean(F.col('defenders_between_ball_goal')), 2).alias('avg_defenders_between_ball_goal'),
        F.round(F.mean(F.col('progression_distance')), 2).alias('avg_progression_distance'),
        F.round(F.mean(F.col('total_players_between_ball_goal')), 2).alias('avg_total_players_between_ball_goal'),
        F.round(F.mean(F.col('atk_def_advantage_between_ball_goal')), 2).alias('avg_atk_def_advantage_between_ball_goal'),
        F.round(F.mean(F.col('progression_distance_norm')), 2).alias('avg_progression_distance_norm'),
        F.round(F.mean(F.col('total_players_between_ball_goal_norm')), 2).alias('avg_total_players_between_ball_goal_norm'),
        F.round(F.mean(F.col('atk_def_advantage_between_ball_goal_norm')), 2).alias('avg_atk_def_advantage_between_ball_goal_norm'),
    )
)

df_avg_threat.show()

In [ ]:
df_avg_threat_match_stats = (
    df_avg_threat.join(
        df_pl_match_stats_22_23_filtrado_mapped,
        on= ['date', 'homeTeamName', 'opponentTeamName'],
        how='left'
    )
)

df_avg_threat_match_stats.show()

In [ ]:
df_pl_match_stats_22_23_filtrado_mapped_home = (
    df_avg_threat_match_stats
    .filter(
        F.col('homeTeam')
        )
    .select(
        'competitionId',
        'season',
        'gameId',
        "date",
        F.col("homeTeamName").alias("teamName"),
        "avg_threat_score",
        'avg_progression_distance_norm',
        'avg_total_players_between_ball_goal_norm',
        'avg_atk_def_advantage_between_ball_goal_norm',
        (F.col("FTR") == 'H').alias('win'),
        F.col("FTHG").alias("goals"),
        F.col("HS").alias("shots"),
        F.col("HST").alias("shots_target"),
        F.col("AvgH").alias("avg_win_odds")
    )
)

df_pl_match_stats_22_23_filtrado_mapped_away = (
    df_avg_threat_match_stats
    .filter(
        ~F.col('homeTeam')
    )
    .select(
        'competitionId',
        'season',
        'gameId',
        "date",
        F.col("opponentTeamName").alias("teamName"),
        "avg_threat_score",
        'avg_progression_distance_norm',
        'avg_total_players_between_ball_goal_norm',
        'avg_atk_def_advantage_between_ball_goal_norm',
        (F.col("FTR") == 'A').alias('win'),
        F.col("FTAG").alias("goals"),
        F.col("AS").alias("shots"),
        F.col("AST").alias("shots_target"),
        F.col("AvgA").alias("avg_win_odds")
))

df_team_match = df_pl_match_stats_22_23_filtrado_mapped_home.unionByName(df_pl_match_stats_22_23_filtrado_mapped_away).sort('avg_threat_score', ascending=False)
df_team_match.show(truncate=False)

In [ ]:
remove_cols = ['competitionId','season','gameId','date','teamName']

df_team_match_pd = df_team_match.toPandas()
df_team_match_pd.drop(remove_cols, axis=1).corr()